In [1]:
import os
%pwd
os.chdir("../")
%pwd

'c:\\Users\\HP\\Documents\\LLM_learning\\End-to-end-ML-with-mlflow'

In [2]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "sahil6398"
os.environ['MLFLOW_TRACKING_PASSWORD'] = "ef0215b239a6d62811942fcaa35ed6655409ca2e"
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/sahil6398/End-to-end-ML-with-mlflow.mlflow"

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metric_file_name: Path
    all_params : dict
    target_column: str
    mlflow_url : str

In [4]:
from ml_project.constants import *
from ml_project.utils.common import *

In [9]:
class ConfigurationManager:
    def __init__(
        self, 
        config_file_path=CONFIG_FILE_PATH, 
        params_file_path=PARAMS_FILE_PATH, 
        schema_file_path=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.Elasticnet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=(config.root_dir),
            test_data_path=(config.test_data_path),
            model_path=(config.model_path),
            metric_file_name=(config.metric_file_name),
            all_params=params,
            target_column=schema.name,
            mlflow_url=os.getenv("MLFLOW_TRACKING_URI")
        )

        return model_evaluation_config

In [6]:
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

c:\Users\HP\Documents\LLM_learning\.sahil_mlflow\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def evaluate_metrics(self,actual,pred):
        rmse = np.sqrt(mean_squared_error(actual,pred))
        r2_square = r2_score(actual,pred)
        mae = mean_absolute_error(actual,pred)

        return rmse, r2_square, mae

    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_y = test_data[self.config.target_column]
        test_x = test_data.drop(self.config.target_column, axis=1)
        mlflow.set_registry_uri(self.config.mlflow_url)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)
            rmse, r2_square, mae = self.evaluate_metrics(test_y, predicted_qualities)
            scores = {"rmse": rmse, "r2_square": r2_square, "mae": mae}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)
            mlflow.log_metric('r2', r2_square)
            mlflow.log_metric('mae', mae)

            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetWineModel")
            else:
                mlflow.sklearn.log_model(model, "model")


In [11]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-03-16 16:26:27,210]: INFO:common:yaml file: config\config.yaml loaded successfully]
[2026-03-16 16:26:27,215]: INFO:common:yaml file: params.yaml loaded successfully]
[2026-03-16 16:26:27,223]: INFO:common:yaml file: schema.yaml loaded successfully]
[2026-03-16 16:26:27,226]: INFO:common:created directory at: artifacts]
[2026-03-16 16:26:27,229]: INFO:common:created directory at: artifacts/model_evaluation]
[2026-03-16 16:26:29,539]: INFO:common:json file saved at: artifacts\model_evaluation\metrics.json]


2026/03/16 16:26:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/16 16:26:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'ElasticnetWineModel' already exists. Creating a new version of this model...
2026/03/16 16:26:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetWineModel, version 2
Created version '2' of model 'ElasticnetWineModel'.


🏃 View run fearless-kite-184 at: https://dagshub.com/sahil6398/End-to-end-ML-with-mlflow.mlflow/#/experiments/0/runs/d20b5daf60ad46dabe715eb0af62022e
🧪 View experiment at: https://dagshub.com/sahil6398/End-to-end-ML-with-mlflow.mlflow/#/experiments/0
